[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR-GITHUB-USERNAME/JAXCode/blob/master/solutions/b_26_mha_pure_solution.ipynb)

# 🟡 Solution: Multi-Head Attention without Flax

*Attention & Transformers · Medium*

Reference implementation. Try it yourself in `b_26_mha_pure.ipynb` first.

---
Problem 06's multi-head attention with an explicit parameter pytree.

### Signature
```python
def init_mha(key, d_model, num_heads):
    ...   # -> nested pytree of the four projections

def apply_mha(params, Q, K, V, num_heads):
    ...   # (B, seq, d_model) x3 -> (B, seq, d_model)
```

### The parameter pytree
```python
{"W_q": {"kernel": (d_model, d_model), "bias": (d_model,)},
 "W_k": {...}, "W_v": {...}, "W_o": {...}}
```

Four projections, each `(d_model, d_model)`, kernels scaled by
`1/sqrt(d_model)` and biases at zero. Split the key **four ways** so the four
kernels are independent — one key reused four times gives four identical
matrices, silently.

### Hyperparameters are not parameters
`num_heads` is an argument to `apply_mha`, not an entry in `params`. A pytree
holds **arrays**; a stray Python int in it becomes a leaf that `jax.grad` will
try to differentiate and `jax.tree.map` will try to scale. Under `jit`,
`num_heads` is static because it determines shapes.

This is the split a module blurs: `self.num_heads` and `self.W_q` look alike
inside a class, but only one of them is a parameter.

### What is unchanged from 06
Everything else. Split into heads, scale by `1/sqrt(d_k)`, softmax over the key
axis, merge, project. Including the trap:

```python
o = o.swapaxes(-3, -2)                    # (..., H, S, d_k) -> (..., S, H, d_k)
o = o.reshape(*o.shape[:-2], d_model)     # only NOW is the reshape correct
```

Use **negative axes and never name the batch**. `vmap` strips the leading axis
off, so a function that unpacks `B, S, D = Q.shape` stops working the moment
anyone maps over it.

reshape does not reorder memory, so `H` and `d_k` have to be adjacent and in
that order before you collapse them.

In [ ]:
# Colab setup (no-op when running locally).
# jax-judge is not published on PyPI, so the judge is installed from the
# repo itself. Regenerate with JAXCODE_REPO=you/YourFork to point this at
# your own fork:  JAXCODE_REPO=you/JAXCode make notebooks
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q flax optax')
    get_ipython().run_line_magic(
        'pip', 'install -q git+https://github.com/YOUR-GITHUB-USERNAME/JAXCode.git')
except ImportError:
    pass

In [ ]:
import jax
import jax.numpy as jnp

print("JAX", jax.__version__, "|", jax.devices())

In [ ]:
# ✅ REFERENCE SOLUTION

import jax
import jax.numpy as jnp


def init_mha(key, d_model, num_heads):
    # Four independent keys: reusing one would give four identical kernels.
    keys = jax.random.split(key, 4)
    return {
        name: {
            "kernel": jax.random.normal(k, (d_model, d_model)) / jnp.sqrt(d_model),
            "bias": jnp.zeros((d_model,)),
        }
        for name, k in zip(("W_q", "W_k", "W_v", "W_o"), keys)
    }


def _dense(p, x):
    return x @ p["kernel"] + p["bias"]


def apply_mha(params, Q, K, V, num_heads):
    d_model = Q.shape[-1]
    d_k = d_model // num_heads

    def heads(t):
        # (..., S, d_model) -> (..., H, S, d_k). H before d_k in the reshape,
        # because d_model is laid out head-major in memory. Never name the
        # leading axes, so this survives vmap stripping the batch away.
        return t.reshape(*t.shape[:-1], num_heads, d_k).swapaxes(-3, -2)

    q = heads(_dense(params["W_q"], Q))
    k = heads(_dense(params["W_k"], K))
    v = heads(_dense(params["W_v"], V))

    scores = jnp.einsum("...hqd,...hkd->...hqk", q, k) / jnp.sqrt(
        jnp.asarray(d_k, Q.dtype)
    )
    o = jnp.einsum("...hqk,...hkd->...hqd", jax.nn.softmax(scores, axis=-1), v)

    # Swap FIRST so H and d_k are adjacent, then collapse them.
    o = o.swapaxes(-3, -2)
    o = o.reshape(*o.shape[:-2], d_model)
    return _dense(params["W_o"], o)

In [ ]:
# 🔍 Verify
import jax
import jax.numpy as jnp

params = init_mha(jax.random.key(0), d_model=8, num_heads=2)
print("pytree:", jax.tree.map(lambda a: a.shape, params))
print("leaves:", len(jax.tree.leaves(params)))

x = jax.random.normal(jax.random.key(1), (2, 5, 8))
print("\nself-attention:", apply_mha(params, x, x, x, 2).shape)

xq = jax.random.normal(jax.random.key(2), (2, 3, 8))
print("cross shapes:  ", apply_mha(params, xq, x, x, 2).shape)

g = jax.grad(lambda p: jnp.sum(apply_mha(p, x, x, x, 2)))(params)
print("\ngrad reaches", len(jax.tree.leaves(g)), "leaves")

In [ ]:
# Run the judge against the reference solution
from jax_judge import check

check("mha_pure")